In [ ]:
%pip install ultralytics

# Cell 1 - Environment check: GPU + ultralytics
import torch, ultralytics

print("torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU - in Colab: Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2 - Get the dataset onto the Colab disk and verify it
import os, shutil, zipfile
from pathlib import Path

ZIP_IN_DRIVE   = "/content/drive/MyDrive/naip_towers_v1.zip" 
EXTRACT_DIR    = Path("/content/_naip_extract")
DATA_ROOT      = Path("/content/naip_towers_v1")

from google.colab import drive

if not DATA_ROOT.exists():
    drive.mount("/content/drive")
    if not os.path.exists(ZIP_IN_DRIVE):
        raise FileNotFoundError(f"Zip not found at {ZIP_IN_DRIVE}")

    print("Extracting", ZIP_IN_DRIVE, "...")

    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

    print("Unzipping...", ZIP_IN_DRIVE)

    with zipfile.ZipFile(ZIP_IN_DRIVE) as zf:
        # replace \\ with /

        for info in zf.infolist():
            info.filename = info.filename.replace("\\", "/")
            zf.extract(info, EXTRACT_DIR)
    
    # The zip may or may not have a top-level naip_towers_v1/ folder; find the dir holding the splits
    root = next(p.parent for p in EXTRACT_DIR.rglob("augmented_train") if p.is_dir())
    shutil.move(str(root), str(DATA_ROOT))
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

# Verify against the counts from the local copy
EXPECTED = {"augmented_train": 9670, "train": 288, "val": 67, "test": 389}


for split, n in EXPECTED.items():
    images = {p.stem for p in (DATA_ROOT / split / "images").glob("*.jpg")}
    labels = {p.stem for p in (DATA_ROOT / split / "labels").glob("*.txt")}

    status = "ok" if len(images) == len(labels) == n and images == labels else "mismatch"

    print(f"{split:16s} images={len(images):5d} labels={len(labels):5d} expected={n:5d}  {status}")
    assert status == "ok", f"{split}: counts or image/label pairing wrong"

# The 72 leaked NY patches must not be in test
leaked = {Path(l.strip()).stem for l in open(DATA_ROOT / "ny_leak_excluded.txt") if l.strip()}
test_stems = {p.stem for p in (DATA_ROOT / "test" / "images").glob("*.jpg")}
assert not (leaked & test_stems), f"{len(leaked & test_stems)} leaked images found in test/"
print(f"Leak check OK: none of the {len(leaked)} excluded files are in test/")

In [ ]:
import yaml

# yaml is used by ultralytics to locate the dataset and class names

DATA_YAML = DATA_ROOT / "data.yaml"
    
data_cfg = {
    "path":  str(DATA_ROOT),
    "train": "augmented_train/images",
    "val":   "val/images",
    "test":  "test/images",
    "names": {0: "tower"},
}
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False))
print(DATA_YAML.read_text())

In [ ]:
# Cell 4 - Train YOLOv8m on augmented_train (checkpoints go to Drive so a disconnect isn't fatal)
import time 
from ultralytics import YOLO

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

RUNS_DIR  = "/content/drive/MyDrive/naip_runs"
RUN_NAME  = "yolov8m"

# the last_ckpt refers to the last checkpoint
last_ckpt = Path(RUNS_DIR) / RUN_NAME / "weights" / "last.pt"

t0 = time.time()

if last_ckpt.exists():
    # A previous session was cut off: pick up from the last finished epoch
    print("Resuming from", last_ckpt)
    model = YOLO(str(last_ckpt))
    model.train(resume=True)
else:
    model = YOLO("yolov8m.pt")  # COCO-pretrained, downloads automatically
    model.train(
        data=str(DATA_YAML),
        epochs=100, 
        patience=20,        # stop early if val mAP hasn't improved in 20 epochs
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=2,          # Colab only has 2 CPU cores
        seed=42,
        deterministic=True,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

print(f"Training wall time this session: {(time.time() - t0) / 3600:.2f} h")

      1/100      11.9G      2.562      3.515      1.895         40        640: 17% ━━────────── 104/605 1.6it/s 1:07<5:16

New https://pypi.org/project/ultralytics/8.4.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/naip_towers_v1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, mom

New https://pypi.org/project/ultralytics/8.4.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/naip_towers_v1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, mom

In [2]:
# Cell 5 - Plot YOLOv8m's three training losses (train vs val) from results.csv
#
# YOLOv8's total loss is a weighted sum of three parts. Lower is better for all three.
#
# box_loss - Bounding-box regression loss (CIoU). Measures how well a predicted box lines up
#            with the real tower box: how much they overlap, how far apart their centers are,
#            and whether their width/height shapes match. High = boxes in the wrong place or
#            the wrong size.
#
# cls_loss - Classification loss (binary cross-entropy). Measures how well the model scores
#            "tower" vs "background" at each location: confident where there IS a tower, low
#            where there isn't. With one class, this is effectively the detection-confidence
#            loss: it drives false positives and missed towers.
#
# dfl_loss - Distribution Focal Loss. YOLOv8 predicts each box edge (left/top/right/bottom) as
#            a probability distribution over possible distances rather than a single number.
#            DFL pushes that distribution to peak sharply at the true edge position. It measures
#            how precisely and confidently the box edges are placed: fine-tuning on top of box_loss.

import csv
import matplotlib.pyplot as plt

RESULTS_CSV = "/content/drive/MyDrive/naip_runs/yolov8m/results.csv"

with open(RESULTS_CSV) as f:
    rows = [{k.strip(): float(v) for k, v in r.items()} for r in csv.DictReader(f)]
epochs = [r["epoch"] for r in rows]

# Epoch best.pt came from: YOLO's fitness score = 0.1*mAP50 + 0.9*mAP50-95 on val
fitness = [0.1 * r["metrics/mAP50(B)"] + 0.9 * r["metrics/mAP50-95(B)"] for r in rows]
best_epoch = epochs[fitness.index(max(fitness))]

TRAIN, VAL = "#2a78d6", "#eb6834"   # blue / orange: stays distinguishable for colorblind readers
LOSSES = [("box_loss", "Box loss (localization)"),
          ("cls_loss", "Class loss (tower vs background)"),
          ("dfl_loss", "DFL loss (box-edge precision)")]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, (key, title) in zip(axes, LOSSES):
    for split, color in (("train", TRAIN), ("val", VAL)):
        y = [r[f"{split}/{key}"] for r in rows]
        ax.plot(epochs, y, color=color, linewidth=2, label=split)
        ax.annotate(split, (epochs[-1], y[-1]), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=9, color="#52514e")
    ax.axvline(best_epoch, color="#8a8984", linestyle="--", linewidth=1)
    ax.set_title(title, fontsize=11, loc="left")
    ax.set_xlabel("Epoch")
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Loss")
axes[0].legend(frameon=False)
fig.suptitle(f"YOLOv8m: {len(rows)} epochs, best.pt from epoch {int(best_epoch)} (dashed line)",
             x=0.01, ha="left", fontsize=13)
fig.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/naip_runs/yolov8m/results.csv'